In [ ]:
import pandas as pd
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

commune = "Chefchaouen"

voies = [
    "Avenue Hassan II",
    "Rue Moulay Ali Ben Rachid",
    "Avenue Sidi Abdelhamid",
    "Quartier Souika",
    "Route Akchour",
    "Avenue Al Wahda",
    "Rue Targui",
    "Quartier El Kharrazine"
]

etapes = ["neuf", "recharg1", "reno_tot"]
materiaux = ["Peinture", "Thermoplastique", "Enduit"]
prestataires = ["Entreprise", "Régie", "entreprise"]

data = []

for i in range(5000):
    row = {
        "nom_commune": commune,
        "nom_voie": random.choice(voies),
        "annee_realisation": random.choice([2019, 2020, 2021, 2022, 2023]),
        "annee_previs": random.choice([2020, 2021, 2022, 2023, 2024]),
        "etape": random.choice(etapes),
        "materiau": random.choice(materiaux),
        "type_prestataire": random.choice(prestataires),
        "longueur": random.randint(50, 500),
        "cout": random.randint(2000, 15000),
        "commentaire": "Travaux marquage routier"
    }
    data.append(row)

df = pd.DataFrame(data)

# -----------------------------
#  Ajouter valeurs manquantes
# -----------------------------

def add_missing(col, percent):
    df.loc[df.sample(frac=percent).index, col] = np.nan

add_missing("annee_realisation", 0.10)  # 10%
add_missing("materiau", 0.20)           # 20% (مهم)
add_missing("cout", 0.25)               # 25% (مهم)
add_missing("longueur", 0.05)           # 5%

# -----------------------------
#  Ajouter outliers
# -----------------------------

# outliers ف cout
df.loc[df.sample(frac=0.01).index, "cout"] = df["cout"] * 10

# outliers ف longueur
df.loc[df.sample(frac=0.01).index, "longueur"] = df["longueur"] * 5

# -----------------------------
#  Ajouter duplications
# -----------------------------

df = pd.concat([df, df.sample(200)])

# -----------------------------
# Sauvegarde
# -----------------------------

df.to_csv("chefchaouen_dataset_final.csv", index=False)


In [ ]:
data=df.copy()

#affiche les 5 promier ligne

In [ ]:
data.head()

#les information de datase

In [ ]:
data.info()

#descreb de data set

In [ ]:
data.describe()

#shape de data set

In [ ]:
row,columns=data.shape
print(f"le nombre de  ligne :{row},le nombre de columns:{columns}")

#detection des valeur manque

In [ ]:
manque=data.isna().sum()*100/len(data)
print(manque)

In [ ]:
data['materiau']=data['materiau'].fillna(data['materiau'].mode()[0])

In [ ]:
from scipy.stats import skew
for col in data.select_dtypes(include='number'):
  skewnes=skew(data[col].dropna())
  manque=data[col].isna().sum()*100/len(data)
  if manque<15 and (-0.5<skewnes<=0.5):
    print(f"{col} distribution symetrique")
    data[col]=data[col].fillna(data[col].mean())
  elif manque<15 and (skewnes<-0.5 or skewnes>0.5):
    print(f"{col} distributionasymethrique")
    data[col]=data[col].fillna(data[col].median())
  else:
    print(f"{col}: utilise une method avencees,la porcentage des valeur manque >15%")

In [ ]:
numerique=data.select_dtypes(include='number')
categorical=data.select_dtypes(include='object')

In [ ]:
#encodage
encoder=OneHotEncoder(sparse_output=False)
x_encoded=encoder.fit_transform(data[['nom_commune','nom_voie','etape','materiau','type_prestataire','commentaire']])
df_encoded=pd.DataFrame(x_encoded,columns=encoder.get_feature_names_out(['nom_commune','nom_voie',
                                                                         'etape','materiau','type_prestataire',
                                                                         'commentaire']),index=data.index)
df=pd.concat([numerique,df_encoded],axis=1)

In [ ]:
df

In [ ]:
#complet/incomplet
complet=df[df['cout'].notnull()]
incomplet=df[df['cout'].isna()]

In [ ]:
x_train = complet.drop(columns=['cout'])
y_train = complet['cout']
x_test = incomplet.drop(columns=['cout'])

In [ ]:
#model
model=RandomForestRegressor(n_estimators=100,random_state=0)
model.fit(x_train,y_train)


In [ ]:
y_predict=model.predict(x_test)

In [ ]:
data.loc[data['cout'].isna(),'cout']=y_predict

In [ ]:
numerique=data.select_dtypes(include='number')
categorical=data.select_dtypes(include='object')

#seperation des donnees

#detection des outlier attraver attraver IQR

In [ ]:
for col in numerique.columns:
  Q1=data[col].quantile(0.25)
  Q3=data[col].quantile(0.75)
  IQR=Q3-Q1
  lower_bar=Q1-1.5*IQR
  upper_bar=Q3+1.5*IQR
  outlier=data[(data[col]<lower_bar)|(data[col]>upper_bar)]
  print(f"{col}:{outlier.shape[0]}")
  plt.figure(figsize=(10,5))
  plt.boxplot(data[col],vert=False)
  plt.title(f'{col}:visualisation de boxplot')
  plt.axvline(lower_bar,linestyle='--',color='blue',label='lower_bar')
  plt.axvline(upper_bar,linestyle='--',color='red',label='upper_bar')
  plt.legend()
  plt.show()


#detection des outlier attraver DBSCAN

In [ ]:
scaler=StandardScaler()
x_scaler=scaler.fit_transform(numerique)
#model
model=DBSCAN(eps=0.3,min_samples=3)
label=model.fit_predict(x_scaler)
data['cluster']=label
outlier=data[data['cluster']==-1]
#visualisation
plt.figure(figsize=(10,5))
plt.scatter(data['annee_previs'],data['annee_realisation'],c=data['cluster'],cmap='rainbow')
plt.title('detection des outlier attraver DBSCAN')
plt.xlabel('annee_previs')
plt.ylabel('annee_realisation')
plt.colorbar(label='cluster')
plt.show()

#detection des outlier attraver z_score

In [ ]:
for col in numerique.columns:
  mean=data[col].mean()
  std=data[col].std()
  z_score=(data[col]-mean)/std
  outlier=abs(z_score)>3
  print(f"{col}:{outlier.sum()}")
  #visulisation attraver histograme
  plt.figure(figsize=(10,5))
  sns.histplot(z_score)
  plt.title(f'{col}:visualisationdes outlier')
  plt.axvline(3, color='red')
  plt.axvline(-3, color='red')

#method de deaux ecart-types

In [ ]:
for col in numerique.columns:
  mean=data[col].mean()
  ecart_types=data[col].std()
  seuil_sepereure=mean+2*ecart_types
  seuil_inferieur=mean-2*ecart_types
  outlier=data[(data[col]>seuil_sepereure)|(data[col]<seuil_inferieur)]
  data.loc[ outlier.index,col]= mean

#isolation forest

In [ ]:

model=IsolationForest(contamination=0.05, random_state=42)
label=model.fit_predict(numerique)
data['cluster']=label
outlier=data[data['cluster']==-1]
print(f"{outlier.shape[0]}")
#visualisation
plt.figure(figsize=(10,5))
plt.scatter(data['annee_realisation'],data['cout'],c=data['cluster'],cmap='rainbow')
plt.title(f'visualisation des outleir')
plt.xlabel('annee_realisation')
plt.ylabel('cout')
plt.colorbar(label='cluster')
plt.show()
for col in numerique.columns:
  median=data[col].median()
  data.loc[data['cluster']==-1,col]=median

In [ ]:
#detection des duplicated
data.duplicated().sum()


In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.isna().sum()*100/len(data)

l'analyse des donnes model GLM

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error,r2_score

In [ ]:
#separation de donnnes
numerique=data.select_dtypes(include='number')
categories=data.select_dtypes(exclude='number')
#encodage
encoder=OneHotEncoder(sparse_output=False)
encoded=encoder.fit_transform(categories)
data_encoded=pd.DataFrame(encoded,columns=encoder.get_feature_names_out(categories.columns),index=data.index)
data_finl=pd.concat([numerique,data_encoded],axis=1)

In [ ]:
#encodage
encoder=OneHotEncoder(sparse_output=False)
encoded=encoder.fit_transform(categories)
data_encoded=pd.DataFrame(encoded,columns=encoder.get_feature_names_out(categories.columns),index=data.index)
data_finl=pd.concat([numerique,data_encoded],axis=1)

In [ ]:
#separation des donnes choise variable cible et variable explicative
x=data_finl.drop(columns=['cout'])
y=data_finl['cout']

In [ ]:
#separatin des donnes
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
#model
model=LinearRegression()
model.fit(x_train,y_train)

In [ ]:
#prediction
y_predict=model.predict(x_test)

In [ ]:
y_predict

In [ ]:
#coef
print(f"la coefision:{model.coef_}")
print(f"l'intercept:{model.intercept_}")

In [ ]:
#performance de model
mse=mean_squared_error(y_test,y_predict)
r2=r2_score(y_test,y_predict)

In [ ]:
data.to_csv("dataset_final_powerbi.csv", index=False)